# Notebook 5A: Introduction to Classification with CalEnviroScreen

*Authored by Dr. Noelle Anderson in 2026*

## Introduction

In Notebook 4B, you fit your first supervised learning model: a **linear regression** model that predicted the numerical outcome `Asthma`.

In this notebook, we will shift to a different kind of supervised learning problem: **classification**.

A **classification** model predicts a category instead of a number. In biology, biochemistry, and public health, classification questions often ask whether a sample, patient, population, or place belongs to one group or another.

For example:

- Is this sample likely to belong to group A or group B?
- Is this cell type, protein variant, or biological sample in one category or another?
- Is this patient, population, or place in a higher-risk or lower-risk category?
- Is this geographic area in a high-health burden or lower-burden category?

We will still use the CalEnviroScreen `Asthma` column, but we will turn it into a new two-category label: **high asthma burden** or **lower asthma burden**.

By the end of this notebook, you should be able to:

- explain the difference between **regression** and **classification**
- explain why **logistic regression** is used for classification even though it has "regression" in the name
- explain why logistic regression is still a **linear model**
- turn a numerical outcome into a binary class label using **binning**
- check **class balance**
- split data for a classification problem
- use `SimpleImputer` to fill in missing feature values after the train/test split
- scale features after imputation
- fit a logistic regression model called `log_model`
- use predicted probabilities and predicted class labels
- explain how a probability threshold connects to class predictions and decision boundaries
- take a cautious first look at logistic regression coefficients
- inspect model fit informally before learning formal classification metrics in Notebook 5B

## Notebook 5A roadmap

In this notebook, we will follow the first classification workflow in three parts.

**Part 1: Create a classification problem**

1. Review regression vs. classification
2. Load the CalEnviroScreen dataset
3. Prepare rows with known `Asthma` values
4. Turn `Asthma` into a binary classification label
5. Check class balance
6. Choose a broader feature set
7. Create `X` and `y`

**Part 2: Preprocess the data and fit logistic regression**

8. Split the rows into training and test sets
9. Check and impute missing feature values using `SimpleImputer`
10. Scale the features
11. Fit a logistic regression model

**Part 3: Inspect probabilities, class predictions, and coefficients**

12. Predict probabilities
13. Predict class labels
14. Connect probabilities to thresholds and decision boundaries
15. Inspect predictions informally
16. Take a cautious look at logistic regression coefficients

# Part 1: Create a classification problem


## Regression vs. classification

In supervised learning, the type of target determines the type of problem.

A **regression** problem predicts a number. In Notebook 4B, the target was the numerical `Asthma` rate, so the model predicted values like 32.4 or 58.7.

A **classification** problem predicts a category. In this notebook, we will create a binary (2 value) category label from `Asthma`:

- `0` will mean lower asthma burden
- `1` will mean high asthma burden

This changes the model question. Instead of asking:

**Can we predict the actual asthma rate?**

We will ask:

**Can we predict whether a census tract is in the high or low asthma burden group?**

Both are supervised learning problems because both use a **known target** column for training. The difference is the *kind* of target we ask the model to predict, whether we're asking for a number or a category label.

## Why is logistic regression used for classification?

The model we will use today is called **logistic regression**.

The name can be confusing. Logistic regression is used for **classification**, even though it has "regression" in the name.

It is called regression because it starts with a regression-like linear equation. The model learns one weight for each feature and combines the feature values into a score. Then it converts that score into a **probability** between 0 and 1.

For a binary classification problem, that probability usually means:

**How likely is this row to belong to class 1?**

In this notebook, class 1 means high asthma burden.

Then the probability is turned into a predicted class label:

- probability below 0.5 becomes predicted class `0`
- probability at or above 0.5 becomes predicted class `1`

The exact cutoff is called a **threshold**. We will use the default threshold of 0.5 today.

## Logistic regression is still a linear model

Logistic regression is a **linear model**, even though it predicts probabilities instead of numerical target values.

That means the model still learns a weighted combination of the input features based on how much they affect the outcome variable. Each feature has a coefficient, and the model combines those weighted feature values into one score. Logistic regression then converts that score into a probability. In this notebook, that probability means the model’s estimated chance that a census tract belongs to class `1`, or the `High Asthma` group.

This has a few implications:

- logistic regression is often a useful first classification model because of how simpler and interpretable it is
- it can be easier to inspect than many more complex models
- it assumes a relatively simple relationship between the features and the classification decision
- it may miss more complex patterns in the data

So logistic regression is a good place to start, but it is still a simplified model. A useful prediction pattern is not the same thing as a complete explanation of why asthma burden differs across communities.

## Step 0: Import the packages we need

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer # New to us!
from sklearn.linear_model import LogisticRegression # New to us!

Today you will see several important `sklearn` verbs again, keep working on learning what these do:

- `fit()` means **learn from data**
- `transform()` means **use what was learned to change data**
- `fit_transform()` means **learn and transform in one step**
- `predict_proba()` means **predict probabilities**
- `predict()` means **predict class labels**

You used `fit()`, `transform()`, and `predict()` in Notebook 4B. Today we will add `SimpleImputer` and `predict_proba()`.

## Step 1: Load the CalEnviroScreen data from Google Drive

We will load the data the same way you did in Notebooks 4A and 4B. Use `Census Tract` as the dataframe index.

### <font color=0D9488>**Question 1:**</font> Load the CalEnviroScreen dataset from the public Google Drive link. Save the dataframe as `ces`, use `Census Tract` as the index column, then print the shape and display the first 5 rows.

In [ ]:
# Data url info saved to variables
file_id = "1X4-6X3VKhR2jRHppI3XuVV79nhHyg8Xb"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

In [ ]:
# Your solution here!

## Step 2: Prepare rows with known `Asthma` values

We will keep the original CalEnviroScreen dataframe intact and choose model features directly later.

For now, the only cleaning step we need is to remove rows where `Asthma` is missing.

In supervised learning, each training row needs a known answer. The model learns by comparing its predictions to the true label, so we cannot train or evaluate on rows where the target value is missing.

Here, we need the original `Asthma` value because we will use it to create the classification label. If we do not know the asthma value for a tract, we cannot decide whether that tract belongs in the high asthma burden group or the lower asthma burden group.

In [ ]:
# Dropping any rows with missing values in Asthma column with dropna
ces_sup_clean = ces.dropna(subset=["Asthma"]).copy()

print("Shape before dropping missing Asthma rows:", ces.shape)
print("Shape after dropping missing Asthma rows:", ces_sup_clean.shape)

## Step 3: Look at the distribution of `Asthma`

Before we create a classification label, we should look at the numerical `Asthma` values.

A **distribution** shows how values are spread out. For `Asthma`, the distribution helps us see whether most tracts have similar asthma rates or whether some tracts have much higher values than others.

In [ ]:
# Instructor-provided plot of the numerical Asthma outcome
plt.figure(figsize=(8, 5))
sns.histplot(data=ces_sup_clean, x="Asthma", bins=30)

# Calculate the median Asthma value
asthma_median = ces_sup_clean["Asthma"].median()

# Add a vertical line at the median
plt.axvline(asthma_median, linestyle="--", color="red")

plt.title("Distribution of Asthma Rates Across Census Tracts with Median marked")
plt.xlabel("Asthma emergency department visit rate")
plt.ylabel("Number of census tracts")
plt.legend()
plt.show()

print("Asthma summary:")
display(ces_sup_clean["Asthma"].describe())

This plot shows the original numerical `Asthma` variable. In Notebook 4B, we used this numerical column directly as a regression target.

For classification, we need categories. That means we will turn the numerical `Asthma` values into a new label.

### <font color=0D9488>**Question 2:**</font> Based on the histogram and summary table, does `Asthma` look evenly spread out, or do some tracts have much higher values than others? Answer in one or two sentences.

**Your solution here!**

## Step 4: Create a binary asthma label using binning

To make a classification problem, we will convert the numerical `Asthma` column into a binary label.

**Binning** means grouping numerical values into categories or "bins." Here, we will create two bins:

- lower asthma burden
- high asthma burden

For this first classification notebook, we will use the **median** asthma value as the cutoff. The median is the middle value after sorting all asthma values.

The median usually creates two groups that are close in size, which makes the first classification workflow easier to understand. It is not a clinical cutoff, biological cutoff, or policy threshold. In a real public health analysis, the cutoff would need to be chosen carefully based on the question, context, and consequences of the classification.

In [ ]:
# Binary label: 1 = high asthma burden, 0 = lower asthma burden
# Using a conditional here on the median
ces_sup_clean["High Asthma"] = (ces_sup_clean["Asthma"] >= asthma_median).astype(int)

print("Median Asthma cutoff:", asthma_median)
display(ces_sup_clean[["Asthma", "High Asthma"]].head(10))

The new `High Asthma` column is our classification target. It is binary, so like asking a question, with 0 as no and 1 as yes.

- `0` means the tract is below the median asthma burden group
- `1` means the tract is at or above the median asthma burden group

This label simplifies a complex public health outcome into two groups. That simplification helps us learn classification, but it also loses information. A tract just barely above the median and a tract far above the median both get label `1`, even though their actual asthma rates may be very different.

## Step 5: Check class balance

**Class balance** means how many rows belong to each class out of those we want to predict.

Recall that for our target:

- class `0` means lower asthma burden
- class `1` means high asthma burden

Class balance matters in ML because a model can behave differently when one class is much more common than the other.

For example, if 95% of rows were class `0`, a model could predict class `0` almost all the time and still look useful at first glance.


### <font color=0D9488>**Question 3:**</font> Use `.value_counts()` to count how many tracts are in each `High Asthma` class. Then use `.value_counts(normalize=True)` to check the proportions.

In [ ]:
# Your solution here!

Because we used the median as the cutoff, the two classes should be close to balanced. That is useful for a first classification notebook because it lets us focus on the workflow before we learn more advanced ideas about imbalanced classification problems.

## Step 6: Choose our feature set

In Notebook 4B, we used a smaller environmental-only feature set so the first regression model would be easier to interpret. In this notebook, we will use a broader feature set with environmental burden variables, community-condition variables, and age-structure variables.

We are expanding the feature set because asthma burden is shaped by more than pollution measurements alone. At the same time, these variables still need careful interpretation. Poverty, education, housing burden, linguistic isolation, and unemployment are not simple individual-level causes of asthma burden. They often reflect broader systems, including racism, segregation, labor conditions, disinvestment, unequal access to care, housing policy, and political power.

Some columns remain in the dataframe but are not part of today’s `model_features`. Context columns identify places, and columns like `Asthma Pctl`, `Pop. Char. Score`, and `Draft CES 5.0 Score` are too closely tied to asthma or summarize multiple parts of the dataset.

Race/ethnicity percentage columns are also not part of today’s `model_features`. These columns matter for environmental justice because racialized patterns can reflect racism, segregation, environmental policy, housing policy, labor conditions, and unequal access to resources and protection. These structural conditions can contribute to asthma burden, but race itself is not a biological cause of asthma.

We will use the feature set below for today’s fitted model.

In [ ]:
# Instructor-provided feature set for this first classification model
model_features = [
    "Ozone",
    "PM2.5",
    "Diesel PM",
    "Drinking Water",
    "Pesticides",
    "Traffic",
    "Cleanup Sites",
    "Education",
    "Housing Burden",
    "Linguistic Isolation",
    "Poverty",
    "Unemployment",
    "Children < 10 years (%)",
    "Elderly > 64 years (%)"
]

print("Number of features:", len(model_features))
print(model_features)

## Step 7: Create `X` and `y`

Now we will separate the data by column role.

- `X` will contain only the selected columns in `model_features`
- `y` will contain the classification target, `High Asthma`

This is the same `X` and `y` structure you used in Notebooks 4A and 4B, but now `y` is a category label instead of a numerical asthma rate. Columns that are not listed in `model_features` remain in the dataframe, but they are not used as inputs to this model.

### <font color=0D9488>**Question 4:**</font> Create the `X` feature dataset using `model_features` and create the `y` target labels using the `High Asthma` column. Then print the shapes of `X` and `y`, and display the first 5 rows of `X`.


In [ ]:
# Your solution here!

# Part 2: Preprocess the datasets and fit a logistic regression model


## Step 8: Split the rows into training and test sets

Now we will split the rows into training and test sets. For classification, we will add one new option you haven't seen yet: `stratify=y`.

**Stratify** as a word just means to arrange or divide things into distinct groups or layers (strata)

Here, adding the `stratify=y` option means that we ask `train_test_split()` to keep the class proportions similar in the training set and the test set.

This is useful in classification because we do not want one split to accidentally get many more high asthma tracts than the other.

Here, this is not a major issue because the full dataset is already close to balanced. Still, `stratify=y` is worth learning because it is especially useful for datasets with stronger class imbalance, like a dataset where 90% of rows are lower asthma burden and 10% are high asthma burden.

The code below is provided because `stratify` is new.

In [ ]:
# Instructor-provided classification train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # Notice stratify=y here!
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
print("Training class proportions:")
display(y_train.value_counts(normalize=True))

print("Test class proportions:")
display(y_test.value_counts(normalize=True))

## Step 9: Check missing feature values

The broader feature set includes some columns with missing values. This is expected in many real datasets.

Missing values in `X` are different from missing values in `y`:

- if the target `y` is missing, we do not know the answer for that row
- if a feature in `X` is missing, we may still be able to keep the row by filling in the missing feature value during preprocessing

Before we fit a model, let's check how many missing values are in the training and test features.

### <font color=0D9488>**Question 5:**</font> Use `.isna().sum()` to count missing values in `X_train` and `X_test`.

In [ ]:
# Your solution here!

Hmmm it looks like now with this expanded feature set, we do have some missing values we need to handle before we can model since simple linear models cannot handle missing values.

## Step 10: Impute missing feature values with `SimpleImputer`

**Imputation** means filling in missing values using a chosen rule.

Today, we will use the `SimpleImputer` tool from `sklearn`. The word "simple" is important. `SimpleImputer` uses straightforward rules, like filling missing values with the median, mean, most common value, or a constant.

For this notebook, we will use the **median** of each feature.

Median imputation is a practical first method because:

- it is easy to understand
- it keeps rows that have only a few missing feature values
- it is less affected by very large or very small values than the mean

But it also has limitations:

- it does not recover the true missing values
- it can hide meaningful missingness patterns
- it treats missing values in a fairly mechanical way
- it does not explain why the values are missing

More complex and sensitive missing-data methods exist, and even `SimpleImputer` has other options. For now, median imputation will be fine.

Notice the order we do things below. We first fit the imputer on `X_train` only, then use that same imputer to transform both `X_train` and `X_test`. This prevents the test rows from influencing the median values learned during preprocessing (data leakage).

In [ ]:
# 1. Initialize an empty imputer object with the settings, here saying we want it to use the median
imputer = SimpleImputer(strategy="median")

Now we will use the supervised preprocessing pattern.

`fit_transform(X_train)` does two things at once:

1. `fit`: learn the median of each feature from the training data
2. `transform`: use those learned medians to fill missing values in the training data

For the test data, we only use `transform(X_test)`. We do **not** fit the imputer on the test data, because the test set is supposed to act like new, unseen data.

In [ ]:
# 2. Fit the imputer on the training data only, then transform the training data
X_train_imputed = imputer.fit_transform(X_train)

# 3. Transform the test data using only the medians learned from X_train
X_test_imputed = imputer.transform(X_test)

# 4. Convert the arrays back into dataframes so we keep column names and row indices
X_train_imputed = pd.DataFrame(
    X_train_imputed,
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed,
    columns=X_test.columns,
    index=X_test.index
)

print("Missing values after imputation in X_train:", X_train_imputed.isna().sum().sum())
print("Missing values after imputation in X_test:", X_test_imputed.isna().sum().sum())

After imputation, both feature tables should have 0 missing values. The imputer learned medians from `X_train`, then applied those same median-filling rules to both the training and test features.

## Step 11: Scale the features

After imputation, we will scale the features.

You saw scaling in Notebook 1B before PCA, and you used it again in Notebook 4B before linear regression. Here, we will use the same supervised-learning rule:

- fit the scaler on the training data only
- transform both training and test data using what the scaler learned from the training data

The only new detail is that we are scaling the **imputed** feature data. That means we will use `X_train_imputed` and `X_test_imputed`, not the original `X_train` and `X_test`.

We scale because logistic regression is a linear model that uses feature values directly. Scaling helps put the features on a more comparable scale and helps the model fit more smoothly.

### <font color=0D9488>**Question 6:**</font> Create a `StandardScaler`. Fit it on `X_train_imputed`, then transform both `X_train_imputed` and `X_test_imputed`. Save the results as `X_train_scaled` and `X_test_scaled`.

In [ ]:
# Your solution here!

The scaler returns NumPy arrays. So like above with imputing, to keep the feature names readable, we will convert the scaled arrays back into pandas dataframes:

In [ ]:
# Instructor-provided conversion back into dataframes for readable column names
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train_imputed.columns,
    index=X_train_imputed.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test_imputed.columns,
    index=X_test_imputed.index
)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

## Step 12: Fit a logistic regression model

Now we are ready to fit our first classification model! We will create a logistic regression model and save it as `log_model`.

In `sklearn`, the workflow should start to be familiar:

1. create the model object
2. use `.fit()` with the training features and training labels


In [ ]:
# Initialize our logistic regression model
log_model = LogisticRegression(max_iter=1000)

# max_iter tells sklearn the maximum number of coefficient-updating steps
# it can use while finding the best-fitting logistic regression model.

### <font color=0D9488>**Question 7:**</font> Fit `log_model` using the scaled training features and the training labels.

In [ ]:
# Your solution here!

When we fit this model, logistic regression learns a set of coefficients. Because it is a linear model, each feature receives a weight. The model uses those weights to calculate a score, then converts that score into a probability.

We will inspect coefficients more carefully later in the course. Today, focus on the classification workflow: probabilities, class labels, and the threshold that turns one into the other.

# Part 3: Inspect probabilities and class predictions


## Step 13: Predict probabilities with `predict_proba()`

Classification models often produce probabilities before they produce class labels.

For logistic regression, `predict_proba()` gives the predicted probability for each class.

Because we have two classes, the output has two columns:

- column 0: predicted probability of class `0`, here lower asthma burden
- column 1: predicted probability of class `1`, here high asthma burden

For this notebook, we care most about column 1 because it tells us the model's predicted probability that a tract belongs to the high asthma burden group.

In [ ]:
# Instructor-provided probability predictions
predicted_probs = log_model.predict_proba(X_test_scaled) # notice the input

# Save the probability of class 1, high asthma burden
high_asthma_probs = predicted_probs[:, 1]

print("First 10 predicted probabilities for high asthma burden:\n")
print(high_asthma_probs[:10])

## Step 14: Convert probabilities into predicted class labels

A probability is not the same thing as a class label like "Sick" or "Healthy".

To turn probabilities into labels, the model uses a **threshold**. The default threshold for many binary classification models is 0.5.

Using a 0.5 threshold:

- probability below 0.5 becomes predicted class `0`
- probability at or above 0.5 becomes predicted class `1`

In this notebook, class `1` means predicted high asthma burden, and class `0` means predicted lower asthma burden.

The threshold is also connected to the idea of a **decision boundary**. A decision boundary is where the model switches from predicting one class to predicting the other. In a simple two-feature problem, we might be able to draw that boundary as a line on a plot. Our model uses many features, so we will not draw the full boundary here, but the basic idea is the same: the model uses the threshold to decide which side of the classification cutoff each row falls on.

### <font color=0D9488>**Question 8:**</font> Using the default threshold of 0.5, translate the 10 predicted probabilities above into class labels.

*Hint: A probability of 0.5 or higher becomes class `1`, which means predicted high asthma burden. A probability below 0.5 becomes class `0`, which means predicted lower asthma burden.*

**Your solution here!**

`predict()` applies this default threshold rule for us:

In [ ]:
# Instructor-provided class predictions
class_preds = log_model.predict(X_test_scaled) # notice the input

prediction_df = pd.DataFrame({
    "Actual class": y_test.values,
    "Predicted probability of high asthma": high_asthma_probs,
    "Predicted class": class_preds
}, index=y_test.index)

display(prediction_df.head(10))

## Step 15: Inspect predictions informally

We are not learning formal classification metrics until Notebook 5B, but we can still inspect the predictions in two simple ways:

1. compare the actual class counts with the predicted class counts
2. look at the distribution of predicted probabilities

In [ ]:
# Instructor-provided informal prediction check

# Count the actual classes in the test set
actual_counts = y_test.value_counts().sort_index()

# Count the model's predicted classes
predicted_counts = pd.Series(class_preds, index=y_test.index).value_counts().sort_index()

# Combine both counts into one table
class_count_comparison = pd.DataFrame({
    "Actual count": actual_counts,
    "Predicted count": predicted_counts
})

display(class_count_comparison)

This table does not tell us everything about model performance, but it helps us notice whether the model is predicting both classes or mostly predicting one class.

In this run, the test set has almost the same number of actual class `0` and class `1` rows. The model is predicting both classes, but it is predicting class `0` a bit more often than class `1`. That suggests the model may be leaning slightly toward the lower-asthma-burden class, even though the actual test labels are close to balanced.

We still do not know from this table which specific rows were classified correctly or incorrectly. We will learn how to check that more carefully in Notebook 5B.

Next, let's look at the distribution of predicted probabilities.

In [ ]:
# Instructor-provided probability plot
plt.figure(figsize=(8, 5))

sns.histplot(high_asthma_probs, bins=20)
plt.axvline(0.5, linestyle="--", color="black")

plt.title("Predicted Probabilities for High Asthma Burden")
plt.xlabel("Predicted probability of class 1: high asthma burden")
plt.ylabel("Number of test rows")
plt.show()

The dashed line marks the 0.5 threshold. Rows to the right of the line are predicted as class `1`, or high asthma burden. Rows to the left are predicted as class `0`, or lower asthma burden.

This plot shows that logistic regression gives us probabilities before class labels. In this run, the predicted probabilities are spread across much of the range from 0 to 1, so the model is not only making predictions close to the 0.5 cutoff.

Probabilities near 0.5 are close to the decision threshold, which means the model is less clearly on one side or the other. Probabilities closer to 0 or 1 are farther from the threshold, which means the model is making a more confident classification.

Confidence is not the same as correctness (in humans too!). A confident prediction can still be wrong, so this is only an informal check.

In Notebook 5B, we will learn classification metrics that let us evaluate these predictions more directly.

## Step 16: A light look at logistic regression coefficients

Because logistic regression is a linear model, it learns one coefficient for each feature, just like the linear regression model you saw in Notebook 4B. These coefficients are part of how the model combines the input features before converting the result into a predicted probability.

Logistic regression coefficients are harder to interpret than linear regression coefficients because they relate to the model’s internal score, often described using log-odds, rather than directly to the predicted probability. We do not need to interpret log-odds in detail here.

For today, we will use the coefficients only as a cautious preview of how the model is using the features. A positive coefficient means higher values of that feature push the model toward predicting class `1`, or high asthma burden, when the other features are also in the model. A negative coefficient means higher values of that feature push the model toward predicting class `0`, or lower asthma burden.

Because we scaled the features, the coefficient sizes are more comparable than they would be on the original feature scales. Still, these coefficients are not causal effects.

In [ ]:
# Instructor-provided coefficient table

# For binary classification, unlike for linear regression,
# coef_ is stored as a 2D array since it also has a class label.
# The [0] gets the first row, which has one coefficient per feature.
log_coefficients = log_model.coef_[0]

# Make a table with feature names and coefficients
coef_df = pd.DataFrame({
    "Feature": model_features,
    "Coefficient": log_coefficients
})

# Use absolute value to sort by coefficient size
coef_df["Absolute coefficient"] = coef_df["Coefficient"].abs()

# Show the largest coefficients first
coef_df = coef_df.sort_values("Absolute coefficient", ascending=False)

display(coef_df)

You will notice this coefficient table looks different from the one in Notebook 4B because both the model question and the feature set changed.

In 4B, linear regression predicted the numerical `Asthma` rate using only environmental burden variables. Here, logistic regression predicts whether a tract is in the `High Asthma` group using environmental burden, community-condition, and age-structure variables.

In this fitted model, positive coefficients push the model more toward predicting class `1`, or high asthma burden. Negative coefficients push the model more toward predicting class `0`, or lower asthma burden. The largest coefficients show which features the model is using most strongly to separate the two classes, but they are still not causal effects.

Several of the largest coefficients in this model come from community-condition variables. That suggests these variables help the model separate high asthma burden tracts from lower asthma burden tracts, and it also shows why expanding the feature set changes interpretation.

Asthma has biological mechanisms, including airway inflammation, immune responses, and environmental triggers. But community-level differences in asthma burden are not explained by simple biological differences between groups.

Variables like education, poverty, unemployment, housing burden, and linguistic isolation often reflect broader structural conditions, including disinvestment, segregation, labor conditions, housing policy, unequal access to care, and unequal political power. These coefficients should be interpreted as model-based patterns that may point toward broader environmental and structural conditions, not as individual blame or simple biological explanations.

# Congratulations, you have completed today's notebook!

## Key Takeaways:

- You learned the difference between regression and classification
- You turned the numerical `Asthma` outcome into a binary classification label by binning at the median
- You checked class balance before modeling
- You used a broader feature set than Notebook 4B
- You split the rows into training and test sets using stratification
- You used `SimpleImputer` to fill missing feature values after the split
- You scaled the features after imputation
- You fit a logistic regression model called `log_model`
- You used `predict_proba()` to get predicted probabilities
- You used `predict()` to get predicted class labels
- You connected probabilities, thresholds, and decision boundaries
- You inspected predictions informally without using formal classification metrics yet
- You took a cautious first look at logistic regression coefficients
- You practiced separating prediction, association, and causal explanation

## Where This Fits in the ML Workflow

This notebook introduced the first full supervised classification workflow:

1. start with rows that have a known target value
2. create a classification label
3. check class balance
4. choose features for this model
5. split into training and test sets
6. impute missing feature values after the split
7. scale features after imputation
8. fit a classification model
9. predict probabilities
10. convert probabilities into class labels
11. inspect results carefully

In Notebook 5B, we will learn formal ways to evaluate classification models. We will introduce tools like accuracy, confusion matrices, precision, recall, and F1 score so we can evaluate classification performance more carefully.